# 02 — FreeCP + SCLIP: T4 kurulumu ve pilot değerlendirme

Bu notebook ayrı bir Python 3.10 ortamı kurar, sabit revizyondaki resmî FreeCP uygulamasını hazırlar ve VOC2012 **train** kümesinden üç görüntü üzerinde orijinal yöntemi değerlendirir.

Üç görüntülük pilot, kurulum ve çıkarım kontrolüdür; makalenin val1449 sonucunun reprodüksiyonu veya yeni yöntemin başarısı olarak yorumlanmaz.

- Donanım: Colab **T4 GPU**.
- Hücreler sırayla yürütülür. İlk kurulum, bağımlılıklar, VOC arşivi ve model ağırlıkları için birkaç GB indirme gerektirir.
- Model işlemleri ayrı Python 3.10 ortamında çalışır. Başlangıç GPU kontrolü ana Colab ortamında Torch içe aktarmadan `nvidia-smi` kullanır.
- CUDA ve kütüphane işlemleri, veri indirmeden önce ayrı ortamda doğrulanır. Hata durumunda deney durdurulur; son hücre tanı arşivini oluşturur.
- Teslim sırasında başlangıç GPU kontrolü düzenlenmiştir; bu düzenleme sözdizimi kontrolünden geçmiş, yeni GPU deneyi yapılmamıştır. Arşivlenmiş deney ortamı ve doğrulama kayıtları proje paketindedir.

**AI desteği:** ChatGPT/Codex; ortam hazırlığı, kod, testler ve dokümantasyona yardımcı olmuştur. Deneyler öğrenci tarafından Colab'da yürütülmüştür. Ayrıntılı katkı bildirimi rapor ve README'dedir.

[FreeCP](https://arxiv.org/abs/2508.00557) · [PyTorch sürümleri](https://pytorch.org/get-started/previous-versions/) · [MMCV paketleri](https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html) · [VOC veri kümesi](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/)


## 1. Çalışma alanı ve GPU kontrolü

Her başlatmada ayrı bir sonuç klasörü oluşturulur. Varsayılan pilot boyutu `N_PILOT = 3` değeridir; kayıtlı ana deneyler ilgili aşama paketleriyle yürütülür.


In [ ]:
import sys, os, json, shutil, subprocess, hashlib
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path('/content/freecp_t4')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
ART = ROOT / 'runs' / RUN_ID
SCRIPTS = ART / 'scripts'
REPO = ROOT / 'FreeCP'
ENV = ROOT / 'env_py310'
PY = ENV / 'bin/python'
BOOT = ROOT / 'bootstrap'
DATA = ROOT / 'data'
N_PILOT = 3
SCRIPTS.mkdir(parents=True, exist_ok=True)
BUNDLE = json.loads('{"VERIFICATION.txt": "{\\n  \\"checked_at\\": \\"2026-11-09\\",\\n  \\"host_report_observed\\": {\\n    \\"gpu\\": \\"Tesla T4\\",\\n    \\"memory_gib\\": 14.56,\\n    \\"cuda_kernel_test_passed\\": true\\n  },\\n  \\"assistant_validation\\": {\\n    \\"helper_unit_tests_passed\\": 7,\\n    \\"notebook_syntax_checked\\": true,\\n    \\"isolated_environment_installed\\": false,\\n    \\"full_gpu_inference_executed\\": false\\n  },\\n  \\"note\\": \\"User GPU report verifies host environment only; notebook checks the separate legacy environment before dataset download.\\"\\n}", "check_runtime.py": "\\"\\"\\"Check actual old-runtime CUDA kernels and imports; no model weights needed.\\"\\"\\"\\nimport importlib.metadata\\nimport json\\nimport platform\\nimport subprocess\\nimport sys\\nimport traceback\\nfrom pathlib import Path\\nfrom stage2_helpers import UPSTREAM_SHA, save_json\\n\\n\\ndef check(repo, output):\\n    report = {\'ok\': False, \'python\': platform.python_version(), \'checks\': {}, \'packages\': {}}\\n    try:\\n        if sys.version_info[:2] != (3, 10):\\n            raise RuntimeError(\'This isolated recipe requires Python 3.10.\')\\n        report[\'commit\'] = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\\n        if report[\'commit\'] != UPSTREAM_SHA:\\n            raise RuntimeError(\'Unexpected FreeCP source revision.\')\\n        for name in [\'torch\', \'torchvision\', \'numpy\', \'mmcv\', \'mmengine\', \'mmsegmentation\']:\\n            report[\'packages\'][name] = importlib.metadata.version(name)\\n        import torch\\n        report[\'cuda_available\'] = torch.cuda.is_available()\\n        report[\'torch_cuda_build\'] = torch.version.cuda\\n        if not report[\'cuda_available\']:\\n            raise RuntimeError(\'CUDA is unavailable in the isolated environment.\')\\n        report[\'gpu\'] = torch.cuda.get_device_name(0)\\n        report[\'gpu_memory_gib\'] = torch.cuda.get_device_properties(0).total_memory / 1024**3\\n        x = torch.ones((32, 32), device=\'cuda\', dtype=torch.float16)\\n        if (x @ x).mean().item() != 32.0:\\n            raise RuntimeError(\'CUDA FP16 matrix check failed.\')\\n        report[\'checks\'][\'cuda_fp16_matmul\'] = True\\n        import torchvision\\n        boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], device=\'cuda\')\\n        scores = torch.tensor([.9, .8], device=\'cuda\')\\n        if torchvision.ops.nms(boxes, scores, .5).tolist() != [0]:\\n            raise RuntimeError(\'torchvision CUDA extension check failed.\')\\n        report[\'checks\'][\'torchvision_cuda_nms\'] = True\\n        from mmcv.ops import nms\\n        _, indices = nms(boxes, scores, .5)\\n        if indices.tolist() != [0]:\\n            raise RuntimeError(\'MMCV CUDA extension check failed.\')\\n        torch.cuda.synchronize()\\n        report[\'checks\'][\'mmcv_cuda_nms\'] = True\\n        sys.path.insert(0, str(Path(repo).resolve()))\\n        import freecp_segmentor\\n        import custom_datasets\\n        from open_clip import tokenizer\\n        tokens = tokenizer.tokenize([\'a photo of a dog\'])\\n        report[\'checks\'][\'freecp_import\'] = True\\n        report[\'checks\'][\'tokenizer_shape\'] = list(tokens.shape)\\n        if list(tokens.shape) != [1, 77]:\\n            raise RuntimeError(\'Unexpected tokenizer output.\')\\n        report[\'ok\'] = True\\n    except Exception as exc:\\n        report[\'error\'] = f\'{type(exc).__name__}: {exc}\'\\n        report[\'traceback\'] = traceback.format_exc()\\n        raise\\n    finally:\\n        save_json(output, report)\\n        print(json.dumps(report, indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    check(Path(sys.argv[1]), Path(sys.argv[2]))\\n", "first_check.py": "\\"\\"\\"Phase-zero checks; standard library only. This does NOT run segmentation.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport importlib.metadata\\nimport importlib.util\\nimport json\\nimport math\\nimport platform\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nUPSTREAM_SHA = \\"6946aa7999ec5d57d1fe91968ba1ae35a1e53295\\"\\nEPS = 1e-8\\n\\n\\ndef published_code_score(before, after):\\n    \\"\\"\\"FreeCP redundancy_purification score, not the paper\'s Eq. 4.\\n\\n    Inputs are flattened nonnegative maps. Matches the source expression:\\n    sum(X*Y) / (sum(X) + sum(Y) + 1e-8).\\n    \\"\\"\\"\\n    if len(before) != len(after) or not before:\\n        raise ValueError(\\"Maps must have equal, non-zero lengths.\\")\\n    if any(not math.isfinite(x) or x < 0 for x in list(before) + list(after)):\\n        raise ValueError(\\"Maps must contain finite nonnegative values.\\")\\n    return sum(x*y for x, y in zip(before, after)) / (sum(before) + sum(after) + EPS)\\n\\n\\ndef cosine_shape_control(before, after):\\n    \\"\\"\\"Scale-invariant diagnostic control. NOT a validated improved method.\\"\\"\\"\\n    norm = math.sqrt(sum(x*x for x in before) * sum(y*y for y in after))\\n    return sum(x*y for x, y in zip(before, after)) / norm if norm else 0.0\\n\\n\\ndef synthetic_probe():\\n    \\"\\"\\"Duplicate every candidate equally; target probability is divided by m.\\n\\n    Refined spatial map is held fixed to isolate the pruning score. This is a\\n    controlled mathematical example, not real vocabulary expansion or mIoU.\\n    \\"\\"\\"\\n    original = [0.9, 0.8, 0.2, 0.1]\\n    refined = [1.0, 0.75, 0.125, 0.0]\\n    rows = []\\n    for copies in [1, 2, 4, 8]:\\n        target = [p / copies for p in original]\\n        score = published_code_score(target, refined)\\n        rows.append({\\n            \\"duplicate_copies_per_original_candidate\\": copies,\\n            \\"total_candidate_entries\\": 2 * copies,\\n            \\"code_rp_score\\": score,\\n            \\"rp_threshold\\": 0.15,\\n            \\"code_keeps_target\\": score > 0.15,\\n            \\"cosine_shape_control\\": cosine_shape_control(target, refined),\\n        })\\n    return rows\\n\\n\\ndef environment_report():\\n    report = {\\n        \\"recorded_at_utc\\": datetime.now(timezone.utc).isoformat(),\\n        \\"python\\": platform.python_version(),\\n        \\"os\\": platform.system(),\\n        \\"machine\\": platform.machine(),\\n        \\"packages\\": {},\\n        \\"cuda_available\\": False,\\n        \\"cuda_kernel_test_passed\\": False,\\n    }\\n    for name in [\\"torch\\", \\"torchvision\\", \\"numpy\\", \\"mmcv\\", \\"mmcv-lite\\", \\"mmengine\\", \\"mmsegmentation\\"]:\\n        try:\\n            report[\\"packages\\"][name] = importlib.metadata.version(name)\\n        except importlib.metadata.PackageNotFoundError:\\n            report[\\"packages\\"][name] = None\\n    if importlib.util.find_spec(\\"torch\\") is None:\\n        report[\\"next_step\\"] = \\"PyTorch is absent; no image-model experiment ran.\\"\\n        return report\\n    try:\\n        import torch\\n        report[\\"torch_cuda_build\\"] = torch.version.cuda\\n        report[\\"cuda_available\\"] = torch.cuda.is_available()\\n        if report[\\"cuda_available\\"]:\\n            props = torch.cuda.get_device_properties(0)\\n            report[\\"gpu_name\\"] = props.name\\n            report[\\"gpu_memory_gib\\"] = round(props.total_memory / 1024**3, 2)\\n            report[\\"gpu_capability\\"] = list(torch.cuda.get_device_capability(0))\\n            x = torch.ones((32, 32), device=\\"cuda\\")\\n            actual = (x @ x).mean().item()\\n            torch.cuda.synchronize()\\n            report[\\"cuda_kernel_test_passed\\"] = actual == 32.0\\n            del x\\n        report[\\"next_step\\"] = (\\n            \\"GPU kernel works. Validate the FreeCP dependency environment next.\\"\\n            if report[\\"cuda_kernel_test_passed\\"] else\\n            \\"Enable a GPU runtime or resolve CUDA compatibility before baseline evaluation.\\"\\n        )\\n    except Exception as exc:\\n        report[\\"gpu_check_error\\"] = f\\"{type(exc).__name__}: {exc}\\"\\n        report[\\"next_step\\"] = \\"Resolve this environment error before baseline evaluation.\\"\\n    return report\\n\\n\\ndef write_report(output_dir=\\"results\\"):\\n    out = Path(output_dir)\\n    out.mkdir(parents=True, exist_ok=True)\\n    rows = synthetic_probe()\\n    report = {\\n        \\"stage\\": \\"phase_0_environment_and_synthetic_diagnostic\\",\\n        \\"upstream_commit\\": UPSTREAM_SHA,\\n        \\"real_images_evaluated\\": 0,\\n        \\"baseline_reproduced\\": False,\\n        \\"improvement_demonstrated\\": False,\\n        \\"novelty_established\\": False,\\n        \\"environment\\": environment_report(),\\n        \\"synthetic_probe\\": rows,\\n        \\"interpretation\\": (\\n            \\"The published code\'s pruning score changes under uniform rescaling of \\"\\n            \\"the pre-refinement map, with the refined map fixed. This does not show \\"\\n            \\"segmentation failure or establish that cosine similarity improves it.\\"\\n        ),\\n    }\\n    (out / \\"first_check_report.json\\").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding=\\"utf-8\\")\\n    with (out / \\"synthetic_probe.csv\\").open(\\"w\\", newline=\\"\\", encoding=\\"utf-8\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=rows[0].keys())\\n        writer.writeheader()\\n        writer.writerows(rows)\\n    return report\\n\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--output-dir\\", default=\\"results\\")\\n    args = parser.parse_args()\\n    print(json.dumps(write_report(args.output_dir), indent=2, ensure_ascii=False))\\n", "prepare_baseline_config.py": "\\"\\"\\"Generate explicit SCLIP+FreeCP VOC21 configs without importing GPU packages.\\"\\"\\"\\nfrom __future__ import annotations\\nimport argparse\\nimport hashlib\\nimport json\\nimport random\\nimport subprocess\\nfrom pathlib import Path\\nfrom first_check import UPSTREAM_SHA\\n\\n\\ndef prepare(repo, data_root, output, mode=\\"pilot\\", count=50, seed=5110):\\n    repo, data_root, output = Path(repo).resolve(), Path(data_root).resolve(), Path(output).resolve()\\n    if mode not in {\\"pilot\\", \\"reproduction\\"}:\\n        raise ValueError(\\"Mode must be pilot or reproduction.\\")\\n    commit = subprocess.check_output([\\"git\\", \\"-C\\", str(repo), \\"rev-parse\\", \\"HEAD\\"], text=True).strip()\\n    if commit != UPSTREAM_SHA:\\n        raise ValueError(f\\"Expected pinned upstream commit {UPSTREAM_SHA}, got {commit}.\\")\\n    base = repo / \\"configs/cfg_voc21.py\\"\\n    if not base.is_file():\\n        raise FileNotFoundError(base)\\n    split = \\"train\\" if mode == \\"pilot\\" else \\"val\\"\\n    split_file = data_root / f\\"ImageSets/Segmentation/{split}.txt\\"\\n    ids = [s.strip() for s in split_file.read_text().splitlines() if s.strip()]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(\\"Duplicate IDs in dataset split.\\")\\n    if mode == \\"pilot\\":\\n        if count < 1 or count > len(ids):\\n            raise ValueError(\\"Pilot count must be within the available training split.\\")\\n        ids = sorted(random.Random(seed).sample(ids, count))\\n    for item in ids:\\n        if Path(item).name != item:\\n            raise ValueError(\\"Expected simple VOC image IDs.\\")\\n        for part, ext in [(\\"JPEGImages\\", \\".jpg\\"), (\\"SegmentationClass\\", \\".png\\")]:\\n            path = data_root / part / (item + ext)\\n            if not path.is_file():\\n                raise FileNotFoundError(path)\\n    output.parent.mkdir(parents=True, exist_ok=True)\\n    generated_split = output.with_suffix(\\".split.txt\\")\\n    split_text = \\"\\\\n\\".join(ids) + \\"\\\\n\\"\\n    generated_split.write_text(split_text, encoding=\\"utf-8\\")\\n    # Upstream base_config.py defaults to GEM; explicitly override it here.\\n    config = (\\n        f\\"_base_ = {str(base)!r}\\\\n\\"\\n        \\"model = dict(model_type=\'SCLIP\')\\\\n\\"\\n        \\"test_dataloader = dict(num_workers=0, persistent_workers=False, \\"\\n        f\\"dataset=dict(data_root={str(data_root)!r}, ann_file={str(generated_split)!r}))\\\\n\\"\\n    )\\n    output.write_text(config, encoding=\\"utf-8\\")\\n    manifest = {\\n        \\"upstream_commit\\": commit, \\"method\\": \\"SCLIP + FreeCP\\", \\"dataset\\": \\"VOC21\\",\\n        \\"mode\\": mode, \\"source_split\\": split, \\"n_images\\": len(ids),\\n        \\"sampling_seed\\": seed if mode == \\"pilot\\" else None,\\n        \\"split_sha256\\": hashlib.sha256(split_text.encode()).hexdigest(),\\n        \\"config\\": str(output), \\"data_root\\": str(data_root),\\n        \\"warning\\": (\\n            \\"Pilot is from training split; do not compare its mIoU to published validation results.\\"\\n            if mode == \\"pilot\\" else\\n            \\"Full validation split; verify weights, dependencies and preprocessing before comparing published results.\\"\\n        ),\\n    }\\n    output.with_suffix(\\".manifest.json\\").write_text(json.dumps(manifest, indent=2), encoding=\\"utf-8\\")\\n    return manifest\\n\\n\\nif __name__ == \\"__main__\\":\\n    p = argparse.ArgumentParser(description=__doc__)\\n    p.add_argument(\\"--repo\\", required=True)\\n    p.add_argument(\\"--data-root\\", required=True)\\n    p.add_argument(\\"--output\\", required=True)\\n    p.add_argument(\\"--mode\\", choices=[\\"pilot\\", \\"reproduction\\"], default=\\"pilot\\")\\n    p.add_argument(\\"--count\\", type=int, default=50)\\n    p.add_argument(\\"--seed\\", type=int, default=5110)\\n    a = p.parse_args()\\n    print(json.dumps(prepare(a.repo, a.data_root, a.output, a.mode, a.count, a.seed), indent=2))\\n", "requirements_t4.txt": "numpy==1.26.4\\nmmengine==0.8.4\\nmmsegmentation==1.1.1\\nopencv-python==4.8.1.78\\nmatplotlib==3.7.2\\nPillow==10.4.0\\nscipy==1.11.4\\nscikit-learn==1.3.2\\nscikit-image==0.22.0\\nftfy==6.2.0\\nregex==2023.8.8\\ntimm==0.4.12\\neinops==0.3.0\\nhuggingface-hub==0.23.0\\ntransformers==4.37.2\\nsafetensors==0.4.3\\ntqdm==4.65.2\\nprettytable==3.10.0\\nyapf==0.40.1\\naddict==2.4.0\\nPyYAML==6.0.1\\npackaging==23.2\\ntermcolor==2.4.0\\n", "run_pilot.py": "\\"\\"\\"Run the original SCLIP+FreeCP pipeline on a fixed VOC train subset.\\"\\"\\"\\nfrom __future__ import annotations\\nimport argparse\\nimport json\\nimport math\\nimport os\\nimport subprocess\\nimport sys\\nimport time\\nimport traceback\\nfrom pathlib import Path\\nfrom stage2_helpers import UPSTREAM_SHA, digest, save_json\\n\\n\\ndef clean_json(value):\\n    if isinstance(value, dict):\\n        return {str(k): clean_json(v) for k, v in value.items()}\\n    if isinstance(value, (list, tuple)):\\n        return [clean_json(v) for v in value]\\n    if hasattr(value, \'item\'):\\n        return clean_json(value.item())\\n    if isinstance(value, float) and not math.isfinite(value):\\n        return None\\n    return value\\n\\n\\ndef main(repo, config, output):\\n    repo, config, output = Path(repo).resolve(), Path(config).resolve(), Path(output).resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    start = time.perf_counter()\\n    report = {\'ok\': False, \'stage\': \'real_image_smoke_test\', \'method\': \'SCLIP + FreeCP\',\\n              \'real_images_evaluated\': 0, \'full_baseline_reproduced\': False,\\n              \'improvement_demonstrated\': False, \'novelty_established\': False,\\n              \'upstream_commit\': UPSTREAM_SHA, \'model_algorithm_changed\': False,\\n              \'note\': \'Small train subset only. Not comparable with published validation scores.\',\\n              \'timing_note\': \'Unwarmed test-step timings include preprocessing and source mask reads; not a latency benchmark.\',\\n              \'per_image\': []}\\n    try:\\n        actual = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\\n        if actual != UPSTREAM_SHA:\\n            raise RuntimeError(\'Unexpected upstream revision.\')\\n        if subprocess.call([\'git\', \'-C\', str(repo), \'diff\', \'--quiet\', \'HEAD\', \'--\']) != 0:\\n            raise RuntimeError(\'Tracked upstream files changed; stop rather than silently alter the baseline.\')\\n        sys.path.insert(0, str(repo))\\n        os.chdir(repo)\\n        import numpy as np\\n        import torch\\n        from PIL import Image\\n        import freecp_segmentor\\n        import custom_datasets\\n        from mmengine.config import Config\\n        from mmengine.hooks import Hook\\n        from mmengine.runner import Runner\\n        cfg = Config.fromfile(str(config))\\n        if cfg.model.model_type != \'SCLIP\':\\n            raise RuntimeError(\'The baseline must explicitly select SCLIP.\')\\n        manifest = json.loads(config.with_suffix(\'.manifest.json\').read_text())\\n        if manifest[\'source_split\'] != \'train\' or manifest[\'mode\'] != \'pilot\':\\n            raise RuntimeError(\'This driver is only for the training-split smoke test.\')\\n        report[\'dataset_manifest\'] = manifest\\n        cfg.work_dir = str(output / \'mmengine\')\\n        cfg.launcher = \'none\'\\n        cfg.randomness = dict(seed=5110, deterministic=False)\\n        cfg.dump(str(output / \'resolved_config.py\'))\\n        report[\'config_sha256\'] = digest(output / \'resolved_config.py\')\\n        report[\'gpu\'] = torch.cuda.get_device_name(0)\\n        runner = Runner.from_cfg(cfg)\\n        palette = [int(v) for color in runner.test_dataloader.dataset.metainfo[\'palette\'] for v in color]\\n        palette = (palette + [0] * 768)[:768]\\n        palette[255 * 3:256 * 3] = [128, 128, 128]\\n        figures = output / \'figures\'\\n        figures.mkdir(exist_ok=True)\\n\\n        class ArtifactsHook(Hook):\\n            def before_test_iter(self, runner, batch_idx, data_batch=None):\\n                torch.cuda.synchronize()\\n                torch.cuda.reset_peak_memory_stats()\\n                self.started = time.perf_counter()\\n\\n            def after_test_iter(self, runner, batch_idx, data_batch=None, outputs=None):\\n                torch.cuda.synchronize()\\n                elapsed = time.perf_counter() - self.started\\n                peak = torch.cuda.max_memory_allocated() / 1024**3\\n                for sample in outputs:\\n                    image_path = Path(sample.metainfo[\'img_path\'])\\n                    image_id = image_path.stem\\n                    pred = sample.pred_sem_seg.data.squeeze(0).detach().cpu().numpy()\\n                    if pred.ndim != 2 or pred.min() < 0 or pred.max() > 20:\\n                        raise ValueError(\'Unexpected VOC21 prediction labels.\')\\n                    finite = bool(torch.isfinite(sample.seg_logits.data).all().item())\\n                    image = Image.open(image_path).convert(\'RGB\')\\n                    gt_path = Path(manifest[\'data_root\']) / \'SegmentationClass\' / (image_id + \'.png\')\\n                    gt_array = np.array(Image.open(gt_path))\\n                    if pred.shape != gt_array.shape or image.size != (pred.shape[1], pred.shape[0]):\\n                        raise ValueError(\'Prediction and ground-truth spatial dimensions differ.\')\\n                    mask = Image.fromarray(pred.astype(\'uint8\'), mode=\'P\')\\n                    mask.putpalette(palette)\\n                    mask.save(figures / (image_id + \'_prediction.png\'))\\n                    gt = Image.fromarray(gt_array.astype(\'uint8\'), mode=\'P\')\\n                    gt.putpalette(palette)\\n                    preview = Image.new(\'RGB\', (image.width * 3, image.height))\\n                    preview.paste(image, (0, 0))\\n                    preview.paste(gt.convert(\'RGB\'), (image.width, 0))\\n                    preview.paste(mask.convert(\'RGB\'), (image.width * 2, 0))\\n                    preview.save(figures / (image_id + \'_image_gt_prediction.jpg\'))\\n                    row = {\'image_id\': image_id, \'unwarmed_test_step_seconds\': elapsed,\\n                           \'peak_allocated_gpu_gib\': peak, \'finite_logits\': finite,\\n                           \'height\': int(pred.shape[0]), \'width\': int(pred.shape[1]),\\n                           \'predicted_classes\': np.unique(pred).tolist(),\\n                           \'image_sha256\': digest(image_path), \'gt_sha256\': digest(gt_path)}\\n                    report[\'per_image\'].append(row)\\n                    report[\'real_images_evaluated\'] = len(report[\'per_image\'])\\n                    save_json(output / \'pilot_report.json\', clean_json(report))\\n\\n        runner.register_hook(ArtifactsHook())\\n        metrics = runner.test()\\n        report[\'metrics_percent\'] = clean_json(metrics)\\n        if report[\'real_images_evaluated\'] != manifest[\'n_images\']:\\n            raise RuntimeError(\'Evaluated image count differs from the selected split.\')\\n        if not all(row[\'finite_logits\'] for row in report[\'per_image\']):\\n            raise RuntimeError(\'Non-finite logits observed; inspect source behavior before continuing.\')\\n        weights = Path.home() / \'.cache/clip/ViT-B-16.pt\'\\n        if weights.is_file():\\n            report[\'checkpoint\'] = {\'path\': str(weights), \'sha256\': digest(weights)}\\n            if report[\'checkpoint\'][\'sha256\'] != \'5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f\':\\n                raise RuntimeError(\'Unexpected OpenAI ViT-B/16 checkpoint hash.\')\\n        else:\\n            raise RuntimeError(\'Cannot locate the expected checkpoint for provenance.\')\\n        report[\'ok\'] = True\\n    except Exception as exc:\\n        report[\'error\'] = f\'{type(exc).__name__}: {exc}\'\\n        report[\'traceback\'] = traceback.format_exc()\\n        raise\\n    finally:\\n        report[\'total_wall_seconds_including_model_loading\'] = time.perf_counter() - start\\n        save_json(output / \'pilot_report.json\', clean_json(report))\\n        print(json.dumps(clean_json(report), indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    p = argparse.ArgumentParser()\\n    p.add_argument(\'--repo\', required=True)\\n    p.add_argument(\'--config\', required=True)\\n    p.add_argument(\'--output\', required=True)\\n    args = p.parse_args()\\n    main(args.repo, args.config, args.output)\\n", "stage2_helpers.py": "\\"\\"\\"Small standard-library helpers for the isolated Colab experiment.\\"\\"\\"\\nfrom __future__ import annotations\\nimport hashlib\\nimport json\\nimport shutil\\nimport subprocess\\nimport tarfile\\nimport time\\nimport urllib.request\\nfrom pathlib import Path, PurePosixPath\\n\\nUPSTREAM_SHA = \'6946aa7999ec5d57d1fe91968ba1ae35a1e53295\'\\nCLIP_SHA = \'d05afc436d78f1c48dc0dbf8e5980a9d471f35f6\'\\nBPE_GIT_BLOB_SHA = \'7b5088a527f720063f044eb928eee315f63b2fc0\'\\nVOC_MD5 = \'6cd6e144f989b92b3379bac3b3de84fd\'\\nVOC_URL = \'https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar\'\\n\\n\\ndef save_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\\n\\n\\ndef digest(path, algorithm=\'sha256\'):\\n    h = hashlib.new(algorithm)\\n    with Path(path).open(\'rb\') as f:\\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef git_blob_sha(path):\\n    path = Path(path)\\n    h = hashlib.sha1(f\'blob {path.stat().st_size}\\\\0\'.encode())\\n    with path.open(\'rb\') as f:\\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef run_logged(command, label, artifacts, cwd=None, env=None):\\n    \\"\\"\\"Stream output and preserve a stage result, including on process errors.\\"\\"\\"\\n    artifacts = Path(artifacts)\\n    artifacts.mkdir(parents=True, exist_ok=True)\\n    status_path = artifacts / \'stages.json\'\\n    states = json.loads(status_path.read_text()) if status_path.exists() else {}\\n    record = {\'command\': [str(x) for x in command], \'cwd\': str(cwd) if cwd else None,\\n              \'ok\': False, \'started_unix\': time.time()}\\n    states[label] = record\\n    save_json(status_path, states)\\n    try:\\n        with (artifacts / f\'{label}.log\').open(\'w\', encoding=\'utf-8\') as log:\\n            with subprocess.Popen([str(x) for x in command], cwd=cwd, env=env,\\n                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\\n                                  text=True, bufsize=1) as process:\\n                try:\\n                    for line in process.stdout:\\n                        print(line, end=\'\', flush=True)\\n                        log.write(line)\\n                        log.flush()\\n                    record[\'returncode\'] = process.wait()\\n                    record[\'ok\'] = record[\'returncode\'] == 0\\n                except KeyboardInterrupt:\\n                    process.terminate()\\n                    try:\\n                        process.wait(timeout=5)\\n                    except subprocess.TimeoutExpired:\\n                        process.kill()\\n                        process.wait()\\n                    record[\'error\'] = \'Interrupted by user\'\\n                    raise\\n        if not record[\'ok\']:\\n            raise RuntimeError(f\'{label} failed. Preserve this output and run the final ZIP cell.\')\\n    except Exception as exc:\\n        record[\'error\'] = f\'{type(exc).__name__}: {exc}\'\\n        raise\\n    finally:\\n        record[\'wall_seconds\'] = time.time() - record[\'started_unix\']\\n        save_json(status_path, states)\\n\\n\\ndef download(url, destination):\\n    destination = Path(destination)\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    partial = destination.with_name(destination.name + \'.partial\')\\n    request = urllib.request.Request(url, headers={\'User-Agent\': \'FreeCP-course-reproduction/0.2\'})\\n    with urllib.request.urlopen(request, timeout=90) as src, partial.open(\'wb\') as dst:\\n        count, last = 0, 0\\n        while True:\\n            chunk = src.read(1024 * 1024)\\n            if not chunk:\\n                break\\n            dst.write(chunk)\\n            count += len(chunk)\\n            if count - last >= 128 * 1024**2:\\n                print(f\'Downloaded {count / 1024**2:.0f} MiB\', flush=True)\\n                last = count\\n    partial.replace(destination)\\n    return destination\\n\\n\\ndef ensure_tokenizer(repo, artifacts):\\n    repo, artifacts = Path(repo), Path(artifacts)\\n    bpe = repo / \'open_clip/bpe_simple_vocab_16e6.txt.gz\'\\n    url = f\'https://raw.githubusercontent.com/openai/CLIP/{CLIP_SHA}/clip/bpe_simple_vocab_16e6.txt.gz\'\\n    if not bpe.exists():\\n        download(url, bpe)\\n    if git_blob_sha(bpe) != BPE_GIT_BLOB_SHA:\\n        raise ValueError(\'Tokenizer file does not match the pinned OpenAI CLIP Git blob.\')\\n    license_path = artifacts / \'licenses/OpenAI_CLIP_LICENSE.txt\'\\n    download(f\'https://raw.githubusercontent.com/openai/CLIP/{CLIP_SHA}/LICENSE\', license_path)\\n    shutil.copy2(repo / \'LICENSE\', artifacts / \'licenses/FreeCP_LICENSE.txt\')\\n    save_json(artifacts / \'tokenizer_asset.json\', {\\n        \'url\': url, \'clip_commit\': CLIP_SHA, \'git_blob_sha1\': BPE_GIT_BLOB_SHA,\\n        \'sha256\': digest(bpe), \'bytes\': bpe.stat().st_size,\\n        \'note\': \'Missing upstream tokenizer asset supplied; no algorithm change.\'})\\n\\n\\ndef extract_voc_files(archive, destination):\\n    \\"\\"\\"Extract only regular files needed for segmentation; never extract links.\\"\\"\\"\\n    destination = Path(destination).resolve()\\n    allowed = (\'VOCdevkit/VOC2012/JPEGImages/\',\\n               \'VOCdevkit/VOC2012/SegmentationClass/\',\\n               \'VOCdevkit/VOC2012/ImageSets/Segmentation/\')\\n    with tarfile.open(archive, \'r:*\') as tar:\\n        for member in tar:\\n            name = member.name.removeprefix(\'./\')\\n            if not name.startswith(allowed):\\n                continue\\n            path = PurePosixPath(name)\\n            if \'..\' in path.parts or path.is_absolute() or member.issym() or member.islnk():\\n                raise ValueError(f\'Unsafe archive entry: {name}\')\\n            if not member.isfile():\\n                continue\\n            target = destination.joinpath(*path.parts).resolve()\\n            if not target.is_relative_to(destination):\\n                raise ValueError(\'Archive entry escapes destination.\')\\n            target.parent.mkdir(parents=True, exist_ok=True)\\n            with tar.extractfile(member) as src, target.open(\'wb\') as dst:\\n                shutil.copyfileobj(src, dst)\\n\\n\\ndef prepare_voc(data_dir, artifacts):\\n    data_dir, artifacts = Path(data_dir), Path(artifacts)\\n    data_dir.mkdir(parents=True, exist_ok=True)\\n    archive = data_dir / \'VOCtrainval_11-May-2012.tar\'\\n    marker = data_dir / \'voc_verified.json\'\\n    voc = data_dir / \'VOCdevkit/VOC2012\'\\n    if not marker.exists():\\n        if not archive.exists():\\n            if shutil.disk_usage(data_dir).free < 5 * 1024**3:\\n                raise RuntimeError(\'At least 5 GiB free space is needed before downloading VOC.\')\\n            download(VOC_URL, archive)\\n        if digest(archive, \'md5\') != VOC_MD5:\\n            raise ValueError(\'VOC archive checksum mismatch; do not use this archive.\')\\n        extract_voc_files(archive, data_dir)\\n        save_json(marker, {\'url\': VOC_URL, \'archive_md5\': VOC_MD5,\\n                          \'archive_bytes\': archive.stat().st_size})\\n    counts = {}\\n    for split, expected in [(\'train\', 1464), (\'val\', 1449)]:\\n        ids = (voc / f\'ImageSets/Segmentation/{split}.txt\').read_text().split()\\n        if len(ids) != expected or len(ids) != len(set(ids)):\\n            raise ValueError(f\'Unexpected {split} split size or duplicate image IDs.\')\\n        for item in ids:\\n            if Path(item).name != item:\\n                raise ValueError(\'Invalid VOC image ID.\')\\n            for directory, suffix in [(\'JPEGImages\', \'.jpg\'), (\'SegmentationClass\', \'.png\')]:\\n                if not (voc / directory / (item + suffix)).is_file():\\n                    raise FileNotFoundError(item + suffix)\\n        counts[split] = len(ids)\\n    manifest = json.loads(marker.read_text())\\n    manifest.update({\'root\': str(voc.resolve()), \'split_counts\': counts,\\n                     \'source_metadata\': \'torchvision.datasets.voc.DATASET_YEAR_DICT[2012]\',\\n                     \'scope\': \'Original VOC2012 segmentation splits; no augmented labels.\'})\\n    save_json(artifacts / \'dataset_manifest.json\', manifest)\\n    print(json.dumps(manifest, indent=2), flush=True)\\n    return voc\\n\\n\\nif __name__ == \'__main__\':\\n    import argparse\\n    p = argparse.ArgumentParser()\\n    p.add_argument(\'operation\', choices=[\'tokenizer\', \'voc\'])\\n    p.add_argument(\'path\')\\n    p.add_argument(\'artifacts\')\\n    args = p.parse_args()\\n    if args.operation == \'tokenizer\':\\n        ensure_tokenizer(args.path, args.artifacts)\\n    else:\\n        prepare_voc(args.path, args.artifacts)\\n", "test_stage2.py": "import io\\nimport json\\nimport sys\\nimport tarfile\\nimport tempfile\\nimport unittest\\nfrom contextlib import redirect_stdout\\nfrom pathlib import Path\\nfrom unittest.mock import patch\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent))\\nfrom stage2_helpers import extract_voc_files, git_blob_sha, run_logged\\nimport check_runtime\\nimport run_pilot\\n\\n\\nclass Stage2Tests(unittest.TestCase):\\n    def test_git_blob_digest(self):\\n        with tempfile.TemporaryDirectory() as td:\\n            p = Path(td) / \'hello\'\\n            p.write_bytes(b\'hello\\\\n\')\\n            self.assertEqual(git_blob_sha(p), \'ce013625030ba8dba906f756967f9e9ca394464a\')\\n\\n    def test_process_success_is_recorded(self):\\n        with tempfile.TemporaryDirectory() as td, redirect_stdout(io.StringIO()):\\n            run_logged([sys.executable, \'-c\', \'print(\\"ok\\")\'], \'sample\', td)\\n            states = json.loads((Path(td) / \'stages.json\').read_text())\\n            self.assertTrue(states[\'sample\'][\'ok\'])\\n            self.assertEqual((Path(td) / \'sample.log\').read_text().strip(), \'ok\')\\n\\n    def test_process_failure_is_recorded(self):\\n        with tempfile.TemporaryDirectory() as td, redirect_stdout(io.StringIO()):\\n            with self.assertRaises(RuntimeError):\\n                run_logged([sys.executable, \'-c\', \'raise SystemExit(7)\'], \'failed\', td)\\n            state = json.loads((Path(td) / \'stages.json\').read_text())[\'failed\']\\n            self.assertFalse(state[\'ok\'])\\n            self.assertEqual(state[\'returncode\'], 7)\\n\\n    def test_only_needed_dataset_files_are_extracted(self):\\n        with tempfile.TemporaryDirectory() as td:\\n            root = Path(td)\\n            archive = root / \'fixture.tar\'\\n            with tarfile.open(archive, \'w\') as tar:\\n                for name in [\'VOCdevkit/VOC2012/JPEGImages/a.jpg\', \'VOCdevkit/VOC2012/Annotations/a.xml\']:\\n                    info = tarfile.TarInfo(name)\\n                    info.size = 3\\n                    tar.addfile(info, io.BytesIO(b\'abc\'))\\n            extract_voc_files(archive, root / \'out\')\\n            self.assertEqual((root / \'out/VOCdevkit/VOC2012/JPEGImages/a.jpg\').read_bytes(), b\'abc\')\\n            self.assertFalse((root / \'out/VOCdevkit/VOC2012/Annotations\').exists())\\n\\n    def test_archive_traversal_and_links_are_rejected(self):\\n        for is_link in [False, True]:\\n            with self.subTest(is_link=is_link), tempfile.TemporaryDirectory() as td:\\n                root = Path(td)\\n                archive = root / \'fixture.tar\'\\n                with tarfile.open(archive, \'w\') as tar:\\n                    info = tarfile.TarInfo(\'VOCdevkit/VOC2012/JPEGImages/link\' if is_link else \'VOCdevkit/VOC2012/JPEGImages/../../../../escaped\')\\n                    if is_link:\\n                        info.type = tarfile.SYMTYPE\\n                        info.linkname = \'/tmp/outside\'\\n                        tar.addfile(info)\\n                    else:\\n                        info.size = 3\\n                        tar.addfile(info, io.BytesIO(b\'abc\'))\\n                with self.assertRaises(ValueError):\\n                    extract_voc_files(archive, root / \'out\')\\n\\n    def test_runtime_error_writes_readable_report(self):\\n        with tempfile.TemporaryDirectory() as td, redirect_stdout(io.StringIO()):\\n            path = Path(td) / \'runtime.json\'\\n            with patch.object(check_runtime.sys, \'version_info\', (3, 13, 0)):\\n                with self.assertRaisesRegex(RuntimeError, \'Python 3.10\'):\\n                    check_runtime.check(Path(td), path)\\n            report = json.loads(path.read_text())\\n            self.assertFalse(report[\'ok\'])\\n            self.assertIn(\'Python 3.10\', report[\'error\'])\\n\\n    def test_pilot_source_error_never_claims_success(self):\\n        with tempfile.TemporaryDirectory() as td, redirect_stdout(io.StringIO()):\\n            root = Path(td)\\n            with patch(\'run_pilot.subprocess.check_output\', return_value=\'wrong-revision\'):\\n                with self.assertRaisesRegex(RuntimeError, \'revision\'):\\n                    run_pilot.main(root, root / \'config.py\', root / \'result\')\\n            report = json.loads((root / \'result/pilot_report.json\').read_text())\\n            self.assertFalse(report[\'ok\'])\\n            self.assertEqual(report[\'real_images_evaluated\'], 0)\\n            self.assertFalse(report[\'full_baseline_reproduced\'])\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main(verbosity=2)\\n", "README_RUN.txt": "FreeCP + SCLIP / Colab T4 first real-image run\\nThis is a small training-split smoke test, not the final reproduction result.\\nDefault: 3 VOC2012 train images; seed 5110; original VOC21 preprocessing.\\nModel: CLIP ViT-B/16 + SCLIP + unchanged FreeCP RP/AP.\\nSource: https://github.com/chenqi1126/FreeCP/tree/6946aa7999ec5d57d1fe91968ba1ae35a1e53295\\nPaper: https://arxiv.org/abs/2508.00557\\nRuntime: isolated Python 3.10, torch 1.13.1+cu117, torchvision 0.14.1+cu117,\\nmmcv 2.0.1, mmengine 0.8.4, mmsegmentation 1.1.1, numpy 1.26.4.\\nThe current host Python/PyTorch is not downgraded. This recipe targets T4,\\nnot RTX 5070. GPU compatibility must pass the included extension tests.\\nDeviation: curated runtime dependencies, numpy 1.26.4, num_workers=0;\\nexplicit SCLIP instead of the repository\'s default GEM; missing BPE asset\\nsupplied from pinned OpenAI CLIP. No algorithm edits.\\nrun_pilot.py uses the same Runner.from_cfg + runner.test path as eval.py,\\nadding output/measurement hooks and recording a seed. Runtime overhead from\\nexport hooks and cold starts means these times are not a formal benchmark.\\nPreview columns: input image, ground truth, model prediction. Gray = GT void.\\nCode and non-GPU helpers were checked in the assistant\'s CPU environment.\\nThe full dependency install, downloads and inference are not claimed tested\\nuntil this notebook runs successfully on the user\'s T4.\\nAI assistance: ChatGPT/Codex prepared the source audit, environment recipe,\\nnotebook and helper scripts. Student must review code, run experiments,\\nverify measurements, understand and disclose assistance in the final report.\\nDataset access: https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/\\nOfficial archive/checksum metadata: torchvision.datasets.voc, version\\nv0.14.1 and the current official torchvision source. No augmented labels.\\nDependencies and outputs are recorded in this run directory; checkpoints,\\nthe dataset, and credentials are not included in the result ZIP.\\n"}')
for name, content in BUNDLE.items():
    (SCRIPTS / name).write_text(content, encoding='utf-8')
sys.path.insert(0, str(SCRIPTS))
from stage2_helpers import run_logged, save_json, UPSTREAM_SHA
APP_ENV = os.environ.copy()
APP_ENV['MPLBACKEND'] = 'Agg'
APP_ENV['TOKENIZERS_PARALLELISM'] = 'false'

def run(args, label, cwd=None, env=None):
    return run_logged(args, label, ART, cwd=cwd, env=APP_ENV if env is None else env)

# Host Torch is deliberately not imported: Colab may use Python 3.13.
# Actual CUDA/Torch kernels are tested in the isolated Python 3.10 environment.
host = {'python': sys.version, 'free_disk_gib': shutil.disk_usage(ROOT).free / 1024**3}
try:
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, check=True)
    host['nvidia_smi_gpu'] = probe.stdout.strip()
    if 'T4' not in host['nvidia_smi_gpu']:
        raise RuntimeError('Select a T4 GPU runtime before this setup. Actual CUDA tests run in step 4.')
finally:
    save_json(ART / 'host_environment.json', host)
print(json.dumps(host, indent=2))
print('Bu çalıştırmanın sonuç dizini:', ART)


## 2. Sabit kaynak sürümü ve tokenizer dosyası

FreeCP'nin izlenen kaynak dosyaları değiştirilmiyor. Eksik tokenizer dosyası resmî OpenAI CLIP deposundan alınır ve Git blob özeti doğrulanır; ücretli API veya anahtar gerekmez.


In [ ]:
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/chenqi1126/FreeCP.git', REPO], 'source_clone')
else:
    origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/') not in ['https://github.com/chenqi1126/FreeCP.git', 'https://github.com/chenqi1126/FreeCP']:
        raise RuntimeError('Unexpected repository at the working path.')
run(['git', '-C', REPO, 'checkout', '--detach', UPSTREAM_SHA], 'source_pin')
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == UPSTREAM_SHA
run([sys.executable, SCRIPTS / 'stage2_helpers.py', 'tokenizer', REPO, ART], 'tokenizer_asset')
save_json(ART / 'source_revision.json', {'repository':'https://github.com/chenqi1126/FreeCP', 'commit':actual})


## 3. Ayrı Python 3.10 ortamı ve bağımlılıklar

Python ortamı `uv` ile oluşturulur. Torch 1.13.1 / CUDA 11.7 ve bu sürüm ailesine uygun MMCV paketi kullanılır. NumPy 1.26.4 ve YAPF 0.40.1, eski uygulamayla uyumluluk için sabitlenmiştir. Kurulan bağımlılıkların tam listesi kaydedilir.


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--progress-bar', 'off',
     '--target', BOOT, '--upgrade', 'uv==0.9.10'], 'bootstrap_uv')
UV_ENV = APP_ENV.copy()
UV_ENV['PYTHONPATH'] = str(BOOT)
run([sys.executable, '-m', 'uv', 'python', 'install', '3.10'], 'python310', env=UV_ENV)
if not PY.exists():
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ENV], 'create_env', env=UV_ENV)
run([PY, '-m', 'pip', 'install', '--disable-pip-version-check', '--progress-bar', 'off',
     'pip==24.3.1', 'setuptools==69.5.1', 'wheel==0.44.0'], 'packaging_tools')
constraints = ART / 'constraints.txt'
constraints.write_text('torch==1.13.1+cu117\ntorchvision==0.14.1+cu117\nmmcv==2.0.1\n' +
                       (SCRIPTS / 'requirements_t4.txt').read_text(), encoding='utf-8')
run([PY, '-m', 'pip', 'install', '--progress-bar', 'off', '-c', constraints,
     'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117', 'numpy==1.26.4',
     '--extra-index-url', 'https://download.pytorch.org/whl/cu117'], 'torch_install')
mmcv_wheel = 'https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/mmcv-2.0.1-cp310-cp310-manylinux1_x86_64.whl'
run([PY, '-m', 'pip', 'install', '--progress-bar', 'off', '-c', constraints, mmcv_wheel], 'mmcv_install')
run([PY, '-m', 'pip', 'install', '--progress-bar', 'off', '-c', constraints,
     '-r', SCRIPTS / 'requirements_t4.txt'], 'remaining_dependencies')
run([PY, '-m', 'pip', 'check'], 'pip_check')


## 4. Ayrı ortamda CUDA doğrulaması

Kontroller: FP16 matris çarpımı, torchvision ve MMCV CUDA işlemleri, FreeCP ve tokenizer içe aktarımları. Bu kontroller başarılı olmadan veri indirme ve model değerlendirme aşamalarına geçilmez.


In [ ]:
run([PY, SCRIPTS / 'check_runtime.py', REPO, ART / 'runtime_report.json'], 'runtime_check', cwd=REPO)
runtime_report = json.loads((ART / 'runtime_report.json').read_text())
assert runtime_report['ok'] is True
freeze = subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True, env=APP_ENV)
(ART / 'pip_freeze.txt').write_text(freeze, encoding='utf-8')
print('Ayrı ortamda CUDA işlemleri ve FreeCP importları başarılı.')


## 5. VOC2012 verisi

Resmî VOC2012 train/val arşivi indirilir ve torchvision kaynağındaki MD5 ile doğrulanır. Yalnızca segmentasyon için gerekli dizinler çıkarılır. Bu notebook'taki pilot **train** kümesindedir; sonraki deneylerin geliştirme, confirmation ve daha önce incelenmiş val rolleri raporda ayrıca açıklanmıştır.

Aynı çalışma alanında doğrulanmış arşiv yeniden indirilmez. Colab sanal makinesi silinirse kurulum ve veri erişimi yeniden gerekir.


In [ ]:
assert json.loads((ART / 'runtime_report.json').read_text())['ok']
run([PY, SCRIPTS / 'stage2_helpers.py', 'voc', DATA, ART], 'prepare_voc')
VOC = DATA / 'VOCdevkit/VOC2012'
print('VOC hazır:', VOC)


## 6. Üç gerçek görüntüde SCLIP + FreeCP

Model ağırlığı ilk kullanımda indirilir. Kaynak kodun varsayılanı GEM olduğundan yapılandırma açıkça **SCLIP** seçer.
Görüntüler sabit tohumla train listesinden seçilir; temel RP/AP formülleri değiştirilmez.
Bu adım CPU sayısal kontrolünden farklı olarak gerçek model çıkarımı yapar.


In [ ]:
assert json.loads((ART / 'runtime_report.json').read_text())['ok']
assert 1 <= N_PILOT <= 50
CONFIG = ART / 'pilot_config.py'
run([PY, SCRIPTS / 'prepare_baseline_config.py', '--repo', REPO, '--data-root', VOC,
     '--output', CONFIG, '--mode', 'pilot', '--count', str(N_PILOT), '--seed', '5110'], 'prepare_config')
PILOT = ART / 'pilot'
run([PY, SCRIPTS / 'run_pilot.py', '--repo', REPO, '--config', CONFIG, '--output', PILOT],
    'pilot_run', cwd=REPO)


## 7. Pilot çıktıları

Görsel sütunları: girdi görüntüsü, gerçek etiket ve model tahmini. Üç görüntülük mIoU yalnızca işleyiş kontrolüdür. Bu pilotta ısınma uygulanmadan ölçülen süreler, makalenin skorlarıyla veya standart bir hız benchmarkıyla karşılaştırılmaz.


In [ ]:
report_path = ART / 'pilot/pilot_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({k:report.get(k) for k in ['ok', 'real_images_evaluated', 'metrics_percent', 'error']}, indent=2))
    from IPython.display import display, Image as DisplayImage
    for path in sorted((ART / 'pilot/figures').glob('*_image_gt_prediction.jpg')):
        print(path.name)
        display(DisplayImage(filename=str(path), width=1000))
else:
    print('Pilot henüz çalışmadı. Önceki hücrenin hata kaydını sonuç paketiyle paylaş.')


## 8. Sonuç ve tanı arşivi

ZIP; kurulum günlüklerini, ortam bilgisini, yapılandırmayı ve varsa pilot tahmin/ölçümlerini içerir. Veri arşivi ve model ağırlıkları bu dosyaya dahil edilmez. Başarısız çalıştırmalarda da hata tanısı için arşiv oluşturulabilir; hata kayıtları deney sonucu sayılmaz.


In [ ]:
archive = shutil.make_archive(str(ROOT / ('freecp_t4_results_' + RUN_ID)), 'zip', root_dir=ART)
print('Sonuç paketi:', archive)
from google.colab import files
files.download(archive)
